# CIFAR-10N Uncertainty Visualization - 3 Classes

**Simplified version with 3 classes for clearer decision boundaries**

**Strategy: Work with full 10-class dataset, then filter to 3 classes at the end**

**Features:**
- Only 3 classes (airplane, automobile, bird) for clear visualization
- Decision boundaries visible in 2D
- Misclassified samples marked with X
- DINOv2 (768-dim) OR ResNet-10 (512-dim) feature extraction
- Reuses existing uq_classification package

In [ ]:
# Configuration
FEATURE_EXTRACTOR = 'resnet10'  # 'dinov2_vits14' or 'resnet10'
SELECTED_CLASSES = [0, 1, 2]  # airplane, automobile, bird
UNDER_SUPPORTED_CLASS = 2  # bird (epistemic uncertainty)
UNDER_TRAIN_PER_CLASS = 10
REGULAR_TRAIN_PER_CLASS = 50
EPOCHS = 30
MC_SAMPLES = 30
REDUCTION_METHOD = 'umap'
RANDOM_SEED = 42

print(f"✓ Config: {FEATURE_EXTRACTOR}, 3 classes, {EPOCHS} epochs")

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.manifold import TSNE
try:
    import umap
except:
    print("⚠️ UMAP not available, will use t-SNE")

sys.path.insert(0, str(Path.cwd()))
from src.data.cifar10n_loader import CIFAR10NDataset
from uq_classification.models import EmbeddingDataset, EmbeddingDropoutMLP
from uq_classification.utils import set_seed, auto_device

set_seed(RANDOM_SEED)
device = auto_device('auto')
print(f"Device: {device}")

In [ ]:
# Load full CIFAR-10N dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

dataset = CIFAR10NDataset(
    root='./backend/data/cifar10n',
    train=True,
    noise_type='worse_label',
    transform=transform,
    download=True
)

print(f"✓ Full dataset: {len(dataset)} samples")

In [ ]:
# Manual sampling for 3 classes only
rng = np.random.default_rng(RANDOM_SEED)
clean_labels = np.array(dataset.cifar10.targets)
noise_mask = dataset.noise_mask

train_indices = []
eval_clean_indices = []
eval_aleatoric_indices = []
eval_epistemic_indices = []

for cls in SELECTED_CLASSES:
    cls_all = np.where(clean_labels == cls)[0]
    rng.shuffle(cls_all)
    
    if cls == UNDER_SUPPORTED_CLASS:
        # Under-supported: only clean samples for training
        cls_clean = cls_all[~noise_mask[cls_all]]
        train_indices.extend(cls_clean[:UNDER_TRAIN_PER_CLASS].tolist())
        # Epistemic eval: clean samples from under-supported class
        eval_epistemic_indices.extend(cls_clean[UNDER_TRAIN_PER_CLASS:UNDER_TRAIN_PER_CLASS+50].tolist())
    else:
        # Regular: all samples for training
        train_indices.extend(cls_all[:REGULAR_TRAIN_PER_CLASS].tolist())
        # Clean eval: clean samples from well-supported classes
        cls_clean = cls_all[~noise_mask[cls_all]]
        eval_clean_indices.extend(cls_clean[REGULAR_TRAIN_PER_CLASS:REGULAR_TRAIN_PER_CLASS+50].tolist())
        # Aleatoric eval: noisy samples from well-supported classes
        cls_noisy = cls_all[noise_mask[cls_all]]
        eval_aleatoric_indices.extend(cls_noisy[:50].tolist())

print(f"✓ Train: {len(train_indices)}")
print(f"✓ Eval - Clean: {len(eval_clean_indices)}, Aleatoric: {len(eval_aleatoric_indices)}, Epistemic: {len(eval_epistemic_indices)}")

In [ ]:
# Extract features
@torch.no_grad()
def extract_features(dataset, indices, backbone_type, batch_size=32):
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False)
    
    if backbone_type.startswith('dinov2'):
        from src.models.dinov2_backbone import create_dinov2_model
        model = create_dinov2_model(backbone_type, 10, 0.0, False, True).to(device)
        extract_fn = lambda imgs: model.extract_features(imgs.to(device))
    else:
        resnet = models.resnet18(pretrained=True)
        model = nn.Sequential(*list(resnet.children())[:-1]).to(device)
        extract_fn = lambda imgs: model(imgs.to(device)).squeeze(-1).squeeze(-1)
    
    model.eval()
    all_feats, all_labels, all_clean = [], [], []
    
    for imgs, labels, clean, _ in tqdm(loader, desc=f"Extracting {backbone_type}"):
        all_feats.append(extract_fn(imgs).cpu())
        all_labels.append(labels.cpu())
        all_clean.append(clean.cpu())
    
    return torch.cat(all_feats), torch.cat(all_labels), torch.cat(all_clean)

# Extract all
all_idx = train_indices + eval_clean_indices + eval_aleatoric_indices + eval_epistemic_indices
all_feats, all_labels, all_clean = extract_features(dataset, all_idx, FEATURE_EXTRACTOR)

# Remap labels 0,1,2 -> 0,1,2 (already correct for our 3 classes)
n_train = len(train_indices)
train_feats = all_feats[:n_train]
train_labels = all_clean[:n_train]  # Use clean labels for training
eval_feats = all_feats[n_train:]
eval_clean_labels = all_clean[n_train:]

print(f"✓ Features: {all_feats.shape[1]}-dim, train={train_feats.shape}, eval={eval_feats.shape}")

In [ ]:
# Train MLP
train_ds = EmbeddingDataset(train_feats, train_labels)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model = EmbeddingDropoutMLP(train_feats.shape[1], 3, dropout_p=0.3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(feats), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("✓ Training complete!")

In [ ]:
# MC Dropout
mc_probs = model.mc_forward(eval_feats.to(device), n_passes=MC_SAMPLES).cpu()
mean_probs = mc_probs.mean(0)
pred_classes = mean_probs.argmax(1)
pred_ent = -(mean_probs * torch.log(mean_probs + 1e-10)).sum(1)
misclassified = (pred_classes != eval_clean_labels).numpy()

print(f"✓ MC Dropout: {mc_probs.shape}")
print(f"✓ Misclassification: {misclassified.sum()}/{len(misclassified)} ({misclassified.mean()*100:.1f}%)")

In [ ]:
# Dimensionality reduction
all_feats_np = all_feats.numpy()

if REDUCTION_METHOD == 'umap' and 'umap' in sys.modules:
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED)
    reduced = reducer.fit_transform(all_feats_np)
    print(f"✓ Reduced using UMAP")
else:
    if REDUCTION_METHOD == 'umap':
        print("⚠️ UMAP not available, using t-SNE")
    reducer = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30)
    reduced = reducer.fit_transform(all_feats_np)
    print(f"✓ Reduced using t-SNE")

eval_reduced = reduced[n_train:]

In [ ]:
# Visualize
n_clean = len(eval_clean_indices)
n_aleat = len(eval_aleatoric_indices)
eval_type = np.zeros(len(eval_feats), dtype=int)
eval_type[:n_clean] = 0  # Clean
eval_type[n_clean:n_clean+n_aleat] = 1  # Aleatoric
eval_type[n_clean+n_aleat:] = 2  # Epistemic

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['green', 'orange', 'red']
labels = ['Clean', 'Aleatoric', 'Epistemic']
class_names = ['airplane', 'automobile', 'bird']

# Plot 1: Uncertainty types
for i, (c, l) in enumerate(zip(colors, labels)):
    mask = eval_type == i
    correct = mask & ~misclassified
    axes[0].scatter(eval_reduced[correct, 0], eval_reduced[correct, 1], c=c, label=l, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    wrong = mask & misclassified
    if wrong.sum() > 0:
        axes[0].scatter(eval_reduced[wrong, 0], eval_reduced[wrong, 1], c=c, alpha=0.8, s=100, marker='x', linewidths=3)
axes[0].set_title('Uncertainty Types (X = misclassified)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: True classes
for cls in range(3):
    mask = eval_clean_labels.numpy() == cls
    correct = mask & ~misclassified
    axes[1].scatter(eval_reduced[correct, 0], eval_reduced[correct, 1], c=f'C{cls}', label=class_names[cls], alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    wrong = mask & misclassified
    if wrong.sum() > 0:
        axes[1].scatter(eval_reduced[wrong, 0], eval_reduced[wrong, 1], c=f'C{cls}', alpha=0.8, s=100, marker='x', linewidths=3)
axes[1].set_title('True Classes (X = misclassified)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Entropy
scatter = axes[2].scatter(eval_reduced[:, 0], eval_reduced[:, 1], c=pred_ent.numpy(), cmap='YlOrRd', alpha=0.6, s=50)
if misclassified.sum() > 0:
    axes[2].scatter(eval_reduced[misclassified, 0], eval_reduced[misclassified, 1], c='black', alpha=0.8, s=100, marker='x', linewidths=3, label='Misclassified')
axes[2].set_title('Predictive Entropy', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=axes[2])
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Decision boundaries
from scipy.spatial import cKDTree

x_min, x_max = reduced[:, 0].min() - 1, reduced[:, 0].max() + 1
y_min, y_max = reduced[:, 1].min() - 1, reduced[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

tree = cKDTree(reduced)
_, indices = tree.query(grid_pts, k=1)
grid_feats = torch.from_numpy(all_feats_np[indices]).float()

with torch.no_grad():
    model.eval()
    grid_probs = torch.softmax(model(grid_feats.to(device)), 1)
    grid_preds = grid_probs.argmax(1).cpu().numpy()
    grid_conf = grid_probs.max(1)[0].cpu().numpy()

Z_class = grid_preds.reshape(xx.shape)
Z_conf = grid_conf.reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Decision boundaries
axes[0].contourf(xx, yy, Z_class, levels=np.arange(4)-0.5, cmap='tab10', alpha=0.3)
for cls in range(3):
    mask = (eval_clean_labels.numpy() == cls) & ~misclassified
    axes[0].scatter(eval_reduced[mask, 0], eval_reduced[mask, 1], c=f'C{cls}', s=100, edgecolors='black', linewidth=1.5, alpha=0.8, label=class_names[cls])
if misclassified.sum() > 0:
    axes[0].scatter(eval_reduced[misclassified, 0], eval_reduced[misclassified, 1], c='red', s=150, marker='x', linewidths=3, label='Misclassified')
axes[0].set_title(f'Decision Boundaries - 3 Classes ({FEATURE_EXTRACTOR})', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confidence map
axes[1].contourf(xx, yy, Z_conf, levels=20, cmap='RdYlGn', alpha=0.6)
for i, (c, l) in enumerate(zip(colors, labels)):
    mask = (eval_type == i) & ~misclassified
    axes[1].scatter(eval_reduced[mask, 0], eval_reduced[mask, 1], c=c, label=l, s=100, edgecolors='black', linewidth=1.5, alpha=0.8)
if misclassified.sum() > 0:
    axes[1].scatter(eval_reduced[misclassified, 0], eval_reduced[misclassified, 1], c='red', s=150, marker='x', linewidths=3, label='Misclassified')
axes[1].set_title('Confidence Map', fontsize=14, fontweight='bold')
axes[1].legend()
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('Confidence', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✓ Complete!")

## Summary

**Strategy**: Work with full 10-class dataset, manually sample 3 classes, no complex filtering wrapper needed.

**Reused from uq_classification**:
- `EmbeddingDataset`, `EmbeddingDropoutMLP`
- `model.mc_forward()` for MC dropout
- `set_seed()`, `auto_device()`

**Simple & Clean**: No FilteredDataset wrapper, no label remapping issues, straightforward indexing.